# Deep Learning Practical: Drowsiness Detector
**Task:** Classify eyes as Open/Closed using a CNN and save a model for real-time webcam detection.

> Recommended: Google Colab with GPU.  
> Dataset is downloaded automatically using KaggleHub.


In [ ]:
!pip -q install kagglehub tensorflow opencv-python scikit-learn matplotlib


In [ ]:
import os, shutil, random, glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


## 1. Download Dataset


In [ ]:
import kagglehub

dataset_root = Path(kagglehub.dataset_download("kutaykutlu/drowsiness-detection"))
print("Downloaded to:", dataset_root)

# Locate class folders even if Kaggle changes nesting slightly
closed_candidates = [Path(p) for p in glob.glob(str(dataset_root / "**" / "closed_eye"), recursive=True) if Path(p).is_dir()]
open_candidates = [Path(p) for p in glob.glob(str(dataset_root / "**" / "open_eye"), recursive=True) if Path(p).is_dir()]

if not closed_candidates or not open_candidates:
    raise FileNotFoundError("Could not locate closed_eye/open_eye folders in downloaded dataset.")

closed_dir = closed_candidates[0]
open_dir = open_candidates[0]
print("Closed:", closed_dir)
print("Open:", open_dir)
print("Closed images:", len(list(closed_dir.glob("*"))))
print("Open images:", len(list(open_dir.glob("*"))))


## 2. Create Train / Validation / Test Folders


In [ ]:
work = Path("/content/drowsiness_data")
if work.exists():
    shutil.rmtree(work)

for split in ["train", "val", "test"]:
    for cls in ["closed", "open"]:
        (work / split / cls).mkdir(parents=True, exist_ok=True)

def image_files(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".pgm"}
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts]

def split_copy(files, cls, max_per_class=12000):
    files = files.copy()
    random.Random(SEED).shuffle(files)
    files = files[:min(max_per_class, len(files))]
    n = len(files)
    n_train = int(0.70*n)
    n_val = int(0.15*n)
    parts = {
        "train": files[:n_train],
        "val": files[n_train:n_train+n_val],
        "test": files[n_train+n_val:]
    }
    for split, subset in parts.items():
        for i, src in enumerate(subset):
            shutil.copy2(src, work / split / cls / f"{i:06d}{src.suffix.lower()}")
    return {k: len(v) for k,v in parts.items()}

counts_closed = split_copy(image_files(closed_dir), "closed")
counts_open = split_copy(image_files(open_dir), "open")
print("Closed:", counts_closed)
print("Open:", counts_open)


## 3. Load and Preprocess Data


In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    work / "train",
    label_mode="binary",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    work / "val",
    label_mode="binary",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    work / "test",
    label_mode="binary",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Class names:", train_ds.class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 4. Display Sample Images


In [ ]:
plt.figure(figsize=(10,6))
for images, labels in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3,4,i+1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title("open" if int(labels[i].numpy()[0]) == 1 else "closed")
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Build CNN


In [ ]:
augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
], name="augmentation")

model = models.Sequential([
    layers.Input(shape=(64,64,1)),
    layers.Rescaling(1./255),
    augmentation,

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 6. Train


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/drowsiness_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)


## 7. Accuracy and Loss Graphs


In [ ]:
hist = history.history

plt.figure(figsize=(7,4))
plt.plot(hist["accuracy"], label="Train Accuracy")
plt.plot(hist["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(hist["loss"], label="Train Loss")
plt.plot(hist["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 8. Evaluate on Test Set


In [ ]:
model = tf.keras.models.load_model("/content/drowsiness_cnn.keras")
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


## 9. Classification Report and Confusion Matrix


In [ ]:
y_true = np.concatenate([y.numpy().ravel() for _, y in test_ds]).astype(int)
y_prob = model.predict(test_ds, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["closed", "open"]))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["closed", "open"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


## 10. Test a Few Predictions


In [ ]:
for images, labels in test_ds.take(1):
    probs = model.predict(images[:12], verbose=0).ravel()
    plt.figure(figsize=(10,7))
    for i in range(12):
        ax = plt.subplot(3,4,i+1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        pred = "open" if probs[i] >= 0.5 else "closed"
        true = "open" if int(labels[i].numpy()[0]) == 1 else "closed"
        plt.title(f"T:{true} P:{pred}\n{probs[i]:.2f}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 11. Save / Download Model


In [ ]:
from google.colab import files
print("Saved model: /content/drowsiness_cnn.keras")
files.download("/content/drowsiness_cnn.keras")


## Result
The CNN classifies eye images as **Open** or **Closed**.  
For real-time drowsiness detection, the saved model is used with the included `src/webcam_detector.py`.

A persistent sequence of closed-eye predictions increases a score. Once it crosses the threshold, the system displays **DROWSINESS ALERT** and produces an audible warning.
